In [4]:
import gymnasium as gym
import random, numpy as np, torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

class QNet(nn.Module):
    def __init__(self, s, a):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(s,128), nn.ReLU(), nn.Linear(128,a))
    def forward(self,x):
        return self.fc(x)

class RB():
    def __init__(self, size=10000): self.buf = deque(maxlen=size)
    def push(self, *x): self.buf.append(x)
    def sample(self, n):
        b = random.sample(self.buf, n)
        return map(lambda t: torch.tensor(np.array(t), dtype=torch.float32), zip(*b))
    def __len__(self): return len(self.buf)
    
env = gym.make("CartPole")
s_dim, a_dim = env.observation_space.shape[0], env.action_space.n
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

policy = QNet(s_dim, a_dim).to(device)
target = QNet(s_dim, a_dim).to(device)
target.load_state_dict(policy.state_dict())

opt = optim.Adam(policy.parameters(), lr=1e-3)
buf = RB()
gamma = 0.99
eps, eps_min, eps_decay = 1.0, 0.1, 0.995
B = 64

for ep in range(500):
    s,_ = env.reset()
    total = 0
    for _ in range(500):
        a = env.action_space.sample() if random.random()<eps \
            else policy(torch.tensor(s).float().to(device)).argmax().item()
        s2, r, done, trunc, _ = env.step(a)
        buf.push(s,a,r,s2,float(done))
        s = s2; total += r
        
        if len(buf)>B:
            S,A,R,S2,D = buf.sample(B)
            S, S2 = S.to(device), S2.to(device)
            A = A.long().unsqueeze(1).to(device)
            R, D = R.unsqueeze(1).to(device), D.unsqueeze(1).to(device)
            
            q = policy(S).gather(1,A)
            with torch.no_grad():
                y = R + gamma * target(S2).max(1)[0].unsqueeze(1) * (1-D)
                
            opt.zero_grad()
            nn.SmoothL1Loss()(q,y).backward()
            opt.step()
        
        if done or trunc: break

    eps = max(eps_min, eps*eps_decay)
    if ep%10==0: target.load_state_dict(policy.state_dict())
    print(f"Ep: {ep+1}, R: {total}, Epsilon: {eps:.3f}")
    
scores = []
for i in range(10):
    s,_ = env.reset(); tot=0
    while True:
        a = policy(torch.tensor(s).float().to(device)).argmax().item()
        s, r, done, trunc, _ = env.step(a)
        tot += r
        if done or trunc: break
    scores.append(tot)
    print(f"Test {i+1}: {tot}")

print("Avg: ", np.mean(scores))
env.close()

c:\Users\adity\Desktop\College assignments\MML-RL-and-NLP\RL\rl-venv\Lib\site-packages\gymnasium\envs\registration.py:520: UserWarning: WARN: Using the latest versioned environment `CartPole-v1` instead of the unversioned environment `CartPole`.
  logger.warn(


cpu
Ep: 1, R: 15.0, Epsilon: 0.995
Ep: 2, R: 11.0, Epsilon: 0.990
Ep: 3, R: 16.0, Epsilon: 0.985
Ep: 4, R: 27.0, Epsilon: 0.980
Ep: 5, R: 16.0, Epsilon: 0.975
Ep: 6, R: 13.0, Epsilon: 0.970
Ep: 7, R: 25.0, Epsilon: 0.966
Ep: 8, R: 11.0, Epsilon: 0.961
Ep: 9, R: 23.0, Epsilon: 0.956
Ep: 10, R: 36.0, Epsilon: 0.951
Ep: 11, R: 47.0, Epsilon: 0.946
Ep: 12, R: 20.0, Epsilon: 0.942
Ep: 13, R: 18.0, Epsilon: 0.937
Ep: 14, R: 13.0, Epsilon: 0.932
Ep: 15, R: 23.0, Epsilon: 0.928
Ep: 16, R: 15.0, Epsilon: 0.923
Ep: 17, R: 15.0, Epsilon: 0.918
Ep: 18, R: 18.0, Epsilon: 0.914
Ep: 19, R: 12.0, Epsilon: 0.909
Ep: 20, R: 12.0, Epsilon: 0.905
Ep: 21, R: 16.0, Epsilon: 0.900
Ep: 22, R: 11.0, Epsilon: 0.896
Ep: 23, R: 17.0, Epsilon: 0.891
Ep: 24, R: 13.0, Epsilon: 0.887
Ep: 25, R: 15.0, Epsilon: 0.882
Ep: 26, R: 11.0, Epsilon: 0.878
Ep: 27, R: 19.0, Epsilon: 0.873
Ep: 28, R: 16.0, Epsilon: 0.869
Ep: 29, R: 23.0, Epsilon: 0.865
Ep: 30, R: 10.0, Epsilon: 0.860
Ep: 31, R: 19.0, Epsilon: 0.856
Ep: 32, R: 16